# Results dashboard

**Run Cell → Run All** after the pipelines finish. Data is read from **`metrics_csv/`** (next to this `metrics/` folder).

### What is compared?

The **clinical block is the same** for every column below; only the **RNA embedding branch** changes:

| ID | RNA representation |
|----|---------------------|
| **A** | cAE-corrected embedding, PCA → **32** + clinical |
| **B** | Raw scGPT embedding, PCA → **32** + clinical |
| **C** | cAE embedding **full 512-D** + clinical |

---

### What each section shows

1. **Snapshot** — best survival model & best classifier **per RNA line**, plus top public-cohort AUCs.  
2. **Survival** — C-index heatmap (every survival model × RNA line).  
3. **Response** — ROC-AUC heatmap (every classifier × RNA line).  
4. **OOD** — generalisation to external cohorts (same grids as the CSV).  
5. **Batch correction** — whether the CAE improved mixing vs biology (aggregate + cohort silhouettes).

Fold-by-fold values stay in the CSV files only (keeps this notebook short).


In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import HTML, Markdown, display

TARGET_CINDEX = 0.70


def metrics_csv_dir() -> Path:
    cwd = Path.cwd().resolve()
    for d in (cwd / "metrics_csv", cwd.parent / "metrics_csv"):
        if (d / "v4_survival_results.csv").exists():
            return d
    raise FileNotFoundError(
        "Cannot find metrics_csv/v4_survival_results.csv — open repo root and Run All."
    )


def load_csv(name: str) -> pd.DataFrame | None:
    p = METRICS / name
    return pd.read_csv(p) if p.exists() else None


def style_numeric(df: pd.DataFrame, *, title: str, cmap: str = "Blues") -> None:
    """Readable table: caption + 4 decimals + subtle colour on numeric cells."""
    if df is None or df.empty:
        display(Markdown("*No data.*"))
        return
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    fmt = {c: "{:.4f}" for c in num_cols}
    sty = (
        df.style.format(fmt, na_rep="—")
        .set_caption(title)
        .set_table_styles(
            [
                {
                    "selector": "caption",
                    "props": [
                        ("caption-side", "top"),
                        ("text-align", "left"),
                        ("font-size", "1.05em"),
                        ("font-weight", "600"),
                        ("color", "#1a365d"),
                        ("margin-bottom", "0.5rem"),
                    ],
                },
                {"selector": "th", "props": [("text-align", "center")]},
                {"selector": "td", "props": [("text-align", "center")]},
            ]
        )
    )
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if num_cols:
        sty = sty.background_gradient(cmap=cmap, axis=None, subset=num_cols)
    display(sty)


METRICS = metrics_csv_dir()

display(
    HTML(
        "<p style='background:#ebf8ff;border-left:4px solid #3182ce;padding:8px 12px;"
        "color:#2c5282;font-size:14px;'><b>Data folder:</b> <code>"
        + str(METRICS).replace("\\", "/")
        + "</code></p>"
    )
)

surv = load_csv("v4_survival_results.csv")
clf = load_csv("v4_classification_results.csv")
ood = load_csv("v4_ood_pub_results.csv")
board = load_csv("v4_final_leaderboard.csv")


## 1. Snapshot

In [2]:
# ── 1. Snapshot (leaderboard + survival gate) ─────────────────────────────
_SN = (
    ("feature_set", "RNA + clinical line"),
    ("best_surv_model", "Best survival model"),
    ("cv_cindex", "CV C-index (mean)"),
    ("cv_cindex_std", "CV C-index (std)"),
    ("best_resp_model", "Best classifier"),
    ("cv_auc", "CV ROC-AUC (mean)"),
    ("cv_auc_std", "CV ROC-AUC (std)"),
    ("PUB_BLCA_AUC", "PUB BLCA (best clf)"),
    ("PUB_ccRCC_ICI_AUC", "PUB ccRCC ICI"),
    ("PUB_ccRCC_TKI_AUC", "PUB ccRCC TKI"),
)
_TI = (
    ("Feature set", "RNA + clinical line"),
    ("Best survival model", "Best survival model"),
    ("Internal CV C-index", "CV C-index (mean)"),
    ("Internal CV C-index std", "CV C-index (std)"),
    ("Best response model", "Best classifier"),
    ("Internal CV ROC-AUC", "CV ROC-AUC (mean)"),
    ("Internal CV ROC-AUC std", "CV ROC-AUC (std)"),
    ("PUB_BLCA AUC (best clf)", "PUB BLCA (best clf)"),
    ("PUB_ccRCC_ICI AUC", "PUB ccRCC ICI"),
    ("PUB_ccRCC_TKI AUC", "PUB ccRCC TKI"),
)

if board is None or board.empty:
    display(Markdown("*Run the full v4 pipeline to create `v4_final_leaderboard.csv`.*"))
else:
    schema = _TI if "Feature set" in board.columns else _SN
    cols = [k for k, _ in schema if k in board.columns]
    labels = [d for k, d in schema if k in board.columns]
    snap = board[cols].copy()
    snap.columns = labels
    style_numeric(snap, title="Best-performing models inside each RNA line (TRAIN CV + public cohort bests)")

if surv is None or surv.empty:
    pass
else:
    ix = surv["cindex_mean"].idxmax()
    r = surv.loc[ix]
    ok = float(r["cindex_mean"]) >= TARGET_CINDEX
    badge = "#276749" if ok else "#c05621"
    display(
        HTML(
            f"<div style='margin-top:10px;padding:10px 14px;border-radius:6px;background:#f7fafc;"
            f"border:1px solid #e2e8f0;font-size:14px;'>"
            f"<b style='color:{badge};'>{'PASS' if ok else 'Below target'}</b> "
            f"— survival programme target C-index ≥ {TARGET_CINDEX:.2f}. "
            f"Best observed: <b>{float(r['cindex_mean']):.4f}</b> "
            f"<span style='color:#4a5568;'>(line: {r['embedding']}, model: {r['model']})</span>"
            f"</div>"
        )
    )


,RNA + clinical line,Best survival model,CV C-index (mean),CV C-index (std),Best classifier,CV ROC-AUC (mean),CV ROC-AUC (std),PUB BLCA (best clf),PUB ccRCC ICI,PUB ccRCC TKI
0,cAE-PCA32 + Clinical,DeepSurv,0.6534,0.0090,MLP,0.6216,0.0114,—,—,—
1,Raw scGPT-PCA32 + Clinical,DeepSurv,0.6539,0.0108,Stack,0.6358,0.0284,—,—,—
2,cAE-full + Clinical,DeepSurv,0.6553,0.0107,Stack,0.6217,0.0342,—,—,—


## 2. Survival (C-index)

In [3]:
# ── 2. Survival: mean C-index (every model × RNA line) ─────────────────────
if surv is None or surv.empty:
    display(Markdown("*Missing `v4_survival_results.csv`.*"))
else:
    pv = surv.pivot_table(index="model", columns="embedding", values="cindex_mean", aggfunc="first")
    pv.index.name = "Survival model"
    style_numeric(
        pv.reset_index(),
        title="Harrell C-index on PFS — TRAIN, 5-fold CV mean (higher is better). Std & folds: CSV.",
        cmap="Blues",
    )


embedding,Survival model,Raw scGPT-PCA32 + Clinical,cAE-PCA32 + Clinical,cAE-full + Clinical
0,Cox-strat,0.5610,0.5633,—
1,Coxnet,0.6527,0.6491,0.6476
2,DeepSurv,0.6539,0.6534,0.6553
3,GB-Surv,0.6277,0.6320,0.6316
4,RSF,0.6386,0.6400,0.6315
5,Stack,0.6522,0.6525,0.6529
6,XGB-Cox,0.6307,0.6341,0.6388


## 3. Response (ROC-AUC)

In [4]:
# ── 3. Response: mean ROC-AUC (every classifier × RNA line) ────────────────
if clf is None or clf.empty:
    display(Markdown("*Missing `v4_classification_results.csv`.*"))
else:
    pv = clf.pivot_table(index="model", columns="embedding", values="auc_mean", aggfunc="first")
    pv.index.name = "Classifier"
    style_numeric(
        pv.reset_index(),
        title="Binary response ROC-AUC — TRAIN, 5-fold CV mean. F1 & folds: CSV.",
        cmap="Greens",
    )


embedding,Classifier,Raw scGPT-PCA32 + Clinical,cAE-PCA32 + Clinical,cAE-full + Clinical
0,LightGBM,0.6173,0.5988,0.6035
1,LogReg,0.6032,0.5933,0.6020
2,MLP,0.6122,0.6216,0.6176
3,RF,0.6228,0.6098,0.5878
4,Stack,0.6358,0.6187,0.6217
5,XGBoost,0.6254,0.6039,0.6083


## 4. Out-of-distribution

In [5]:
# ── 4. Out-of-distribution response ───────────────────────────────────────
if ood is None or ood.empty:
    display(Markdown("*Missing `v4_ood_pub_results.csv`.*"))
else:
    wide = (
        ood.pivot_table(index=["embedding", "classifier"], columns="pub_dataset", values="roc_auc", aggfunc="first")
        .reset_index()
        .sort_values(["embedding", "classifier"])
    )
    style_numeric(
        wide,
        title="OOD ROC-AUC — global model trained on full TRAIN, evaluated on each PUB cohort",
        cmap="Oranges",
    )


pub_dataset,embedding,classifier,PUB_BLCA,PUB_ccRCC_ICI,PUB_ccRCC_TKI
0,Raw scGPT + Clinical,LightGBM,0.5336,0.8937,0.4765
1,Raw scGPT + Clinical,LogReg,0.6004,0.6557,0.5024
2,Raw scGPT + Clinical,MLP,0.6177,0.6393,0.4539
3,Raw scGPT + Clinical,RF,0.5557,0.6094,0.4677
4,Raw scGPT + Clinical,Stack,0.5743,0.8356,0.4716
5,Raw scGPT + Clinical,XGBoost,0.5448,0.8065,0.4763


## 5. Batch correction

In [6]:
# ── 5. Batch correction (CAE vs raw scGPT latent) ──────────────────────────
path = METRICS / "batch_correction_metrics.json"
if not path.exists():
    display(Markdown("*No `batch_correction_metrics.json` — run `visualizations/visualizations_cae_correction.ipynb`.*"))
else:
    js = json.loads(path.read_text(encoding="utf-8"))
    bcm = js.get("BATCH_CORRECTION_METRICS", {})
    bef = bcm.get("BEFORE", {})
    aft = bcm.get("AFTER", {})
    ch = bcm.get("CHANGE", {})
    summary = pd.DataFrame(
        [
            {
                "Metric": "Batch silhouette (average)",
                "Before (raw scGPT)": bef.get("AvgBATCH_silhouette"),
                "After (cAE)": aft.get("AvgBATCH_silhouette"),
                "Delta": ch.get("AvgBATCH_silhouette_delta"),
            },
            {
                "Metric": "Biology ARI",
                "Before (raw scGPT)": bef.get("AvgBIO_ARI"),
                "After (cAE)": aft.get("AvgBIO_ARI"),
                "Delta": ch.get("AvgBIO_ARI_delta"),
            },
            {
                "Metric": "Biology NMI",
                "Before (raw scGPT)": bef.get("AvgBIO_NMI"),
                "After (cAE)": aft.get("AvgBIO_NMI"),
                "Delta": ch.get("AvgBIO_NMI_delta"),
            },
        ]
    )
    style_numeric(summary, title="Aggregate mixing vs biological conservation on TRAIN", cmap="PuBu")
    pb = bef.get("per_cohort_silhouette") or {}
    pa = aft.get("per_cohort_silhouette") or {}
    cohorts = sorted(set(pb) | set(pa))
    per = pd.DataFrame(
        [
            {"Cohort": c, "Batch silhouette before": pb.get(c), "Batch silhouette after": pa.get(c)}
            for c in cohorts
        ]
    )
    style_numeric(per, title="Batch silhouette by cohort (mixing quality)", cmap="Blues")


,Metric,Before (raw scGPT),After (cAE),Delta
0,Batch silhouette (average),-0.1571,0.8828,1.0399
1,Biology ARI,0.8495,0.9596,0.1101
2,Biology NMI,0.6941,0.8830,0.1889


,Cohort,Batch silhouette before,Batch silhouette after
0,KIRC,-0.2329,0.8587
1,Melanoma,0.3505,0.9512
2,NSCLC,-0.2190,0.9013
